### CONSTANTS

In [ ]:
short_names = {
    'Pòlissa/Póliza/Policy': 'POLICY',
    'Tecnologia/Tecnología/Technology': 'TECHNOLOGY',
    'Diàmetre comptador (cm)/Diámetro contador (cm)/Counter diameter (cm)': 'DIAMETER',
    'Ús/Uso/Use': 'USAGE',
    "Tipus d'habitatge/Tipo de vivienda/Type of housing": 'HOUSING',
    'Data/Fecha/Date': 'DATETIME',
    'Índex de lectura (L/h)/Índice de lectura (L/h)/Reading index (L/h)': 'CONSUMPTION',
}

KMEANS_THRESHHOLD = 2.5
WINDOW_SIZE = 4

### CODE

#### Load data

In [ ]:
import pyarrow.dataset as ds
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler  
import pandas as pd
import numpy as np

file_path = '../data/lectures_horaries_ABD.parquet'

dataset = ds.dataset(file_path, format="parquet")
table = dataset.to_table()
df = table.to_pandas()

Set up weekday and hour of the day. Drop the hour/date column because redundant

In [ ]:
df = df.rename(columns=short_names)
# Convert 'Data/Fecha/Date' column to datetime format
df['DATETIME'] = pd.to_datetime(df['DATETIME'])

df['WEEKDAY'] = df['DATETIME'].dt.dayofweek  # Extract weekday (Monday=0, Sunday=6)
df['HOUR'] = df['DATETIME'].dt.hour

# df = df.drop(columns=['HOUR/DATE'])

Add the flow (gradient of consumption) as a column.

In [ ]:
df['FLOW'] = df.groupby('POLICY')['CONSUMPTION'].diff()
df = df.dropna(subset=["FLOW"])

Remove bad data (gradient is less than 0, it is impossible that water goes backwards)

In [ ]:
negative_policies = df[df['FLOW'] < 0]['POLICY'].values
filtered_df = df[~df['POLICY'].isin(negative_policies)]

filtered_df

#### Compute the average

In [ ]:
average = filtered_df.groupby(['USAGE', 'HOUSING', 'WEEKDAY', 'HOUR'])['FLOW'].mean().reset_index()
std = filtered_df.groupby(['USAGE', 'HOUSING', 'WEEKDAY', 'HOUR'])['FLOW'].std().reset_index()
average['STD'] = std['FLOW']
stats_df = average.rename(columns={"FLOW": "MEAN"})

In [ ]:
# Assuming original_df is your original dataset and stats_df is the dataset containing MEAN and STD

# Merge the original dataset with the summary dataset on the columns 'USAGE', 'HOUSING', 'WEEKDAY', and 'HOUR'
standarized_df = pd.merge(
    filtered_df,
    stats_df[['USAGE', 'HOUSING', 'WEEKDAY', 'HOUR', 'MEAN', 'STD']],
    on=['USAGE', 'HOUSING', 'WEEKDAY', 'HOUR'],
    how='left'  # Use 'left' to ensure no data is lost from the original dataset
)

# Calculate the standardized flow
standarized_df['STANDARDIZED_FLOW'] = (standarized_df['FLOW'] - standarized_df['MEAN']) / (standarized_df['STD'] + (10 ** (-10)))

# Display the result
standarized_df = standarized_df[['POLICY', 'TECHNOLOGY', 'DIAMETER', 'USAGE', 'HOUSING', 'CONSUMPTION','DATETIME', 'WEEKDAY', 'HOUR', 'STANDARDIZED_FLOW']]
standarized_df = standarized_df.rename(columns={'STANDARDIZED_FLOW' : 'FLOW'})

#### Parse data

To have the flow of multiple hours in a single tuple

In [ ]:
def setup_input(df, window_size = 4, output_path=None):
    """
    Optimized function to create a sliding window dataset.
    
    Args:
    - df (pd.DataFrame): Input DataFrame.
    - window_size (int): Size of the sliding window.
    - output_path (str): Path to save the output. If None, returns a DataFrame.
    
    Returns:
    - pd.DataFrame or None: Processed DataFrame if `output_path` is None.
    """
    # Create a generator to yield rows for each window
    def generate_rows():
        for policy, group in df.groupby('POLICY'):
            # Convert columns to NumPy arrays for efficient slicing
            flows = group['FLOW'].to_numpy()
            technology = group['TECHNOLOGY'].to_numpy()
            usage = group['USAGE'].to_numpy()
            housing = group['HOUSING'].to_numpy()
            consumption = group['CONSUMPTION'].to_numpy()
            hour_date = group['DATETIME'].to_numpy()
            weekday = group['WEEKDAY'].to_numpy()
            hours = group['HOUR'].to_numpy()

            # Generate sliding windows
            for i in range(len(flows) - window_size + 1):
                row = {
                    'POLICY': policy,
                    'TECHNOLOGY': technology[i],
                    'USAGE': usage[i],
                    'HOUSING': housing[i],
                    'CONSUMPTION': consumption[i],
                    'DATETIME': hour_date[i],
                    'WEEKDAY': weekday[i],
                    'START_HOUR': hours[i],
                }
                # Add flow values for the window
                for j in range(window_size):
                    row[f'FLOW_{j + 1}'] = flows[i + j]
                yield row

    # Write to file or return as DataFrame
    if output_path:
        pd.DataFrame.from_records(generate_rows()).to_parquet(output_path, index=False)
        print(f"Saved processed data to {output_path}")
        return None
    else:
        return pd.DataFrame.from_records(generate_rows())


In [ ]:
window_standardized_df = setup_input(standarized_df, WINDOW_SIZE)
window_filtered_df = setup_input(filtered_df, WINDOW_SIZE)
# window_filtered_df.to_parquet("../data/lectures_horaries_windowed.parquet")

In [ ]:
window_filtered_df

#### Heuristic Labeling 

In [ ]:
def label_leaks(df, threshold, num_flows):
    """
    Labels each row as a leak if all flow values exceed the given threshold.

    Parameters:
    df (DataFrame): The dataset containing flow columns (e.g., FLOW_1, FLOW_2, ..., FLOW_n).
    threshold (float): The threshold above which flows are considered a leak.
    num_flows (int): The number of flow columns to check (e.g., 4 if the columns are FLOW_1 to FLOW_4).

    Returns:
    DataFrame: The original DataFrame with an added 'LEAK' column.
    """
    # Create a list of the flow columns based on the number of flows
    flow_columns = [f'FLOW_{i+1}' for i in range(num_flows)]
    
    # Check if all flows exceed the threshold in each row
    df['LEAK'] = df[flow_columns].apply(lambda row: all(abs(row) > threshold), axis=1)
    
    return df

# Example usage:
# df = label_leaks(df, threshold=1.5, num_flows=4)


In [ ]:
heuristic_labeled_df = label_leaks(window_standardized_df, 1, WINDOW_SIZE)

In [ ]:
# Count True and False values in the "LEAK" column
leak_counts = heuristic_labeled_df["LEAK"].value_counts()
# Print the counts
print(leak_counts)

In [ ]:
columns = [f'FLOW_{i}' for i in range(1, WINDOW_SIZE + 1)]
window_standardized_df = window_standardized_df.dropna(subset=columns)
window_standardized_df

In [ ]:
merge_columns = ['POLICY', 'TECHNOLOGY', 'USAGE', 'HOUSING', 'CONSUMPTION', 'WEEKDAY', 'START_HOUR', 'DATETIME']

# Merge the datasets on the specified columns, keeping non-standardized flow columns and adding the LEAK column
window_train_df = pd.merge(
    window_filtered_df,
    heuristic_labeled_df[merge_columns + ['LEAK']],
    on=merge_columns,
    how='left'  # 'left' keeps all rows from non_standardized_df
)

window_train_df = window_train_df.drop_duplicates().reset_index(drop=True)

In [ ]:
window_train_df[20:55]

#### Gaussian Mixture Clustering Labeling (sus results)

In [ ]:
import pandas as pd
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler  

In [ ]:
def label_with_gaussian_mixture(df, window_size = WINDOW_SIZE, n_components=2, threshold=0.5):
    """
    Applies Gaussian Mixture Model (GMM) to label data based on probabilities.

    Args:
      df: Pandas DataFrame with features for clustering (e.g., 'FLOW_1', 'FLOW_2', 'FLOW_3').
      n_components: Number of components (clusters) for the GMM.
      threshold: Probability threshold for assigning a label.

    Returns:
      Pandas DataFrame with an additional 'LEAK' column (True for potential leak, False otherwise).
    """

    # Select features for clustering
    columns = [f'FLOW_{i}' for i in range(1, window_size + 1)]
    X = abs(df[columns])

    # Apply Gaussian Mixture Model
    gmm = GaussianMixture(n_components=n_components, random_state=45)
    gmm.fit(X)

    # Get probabilities of belonging to each cluster
    probabilities = gmm.predict_proba(X)

    # Assign labels based on probability threshold
    df['LEAK'] = probabilities[:, 1] > threshold  # Assuming cluster 1 represents potential leaks

    return df
# Assuming you have the 'joined_averages_df' from the previous step

In [ ]:
# Apply Gaussian Mixture for labeling
labeled_df = label_with_gaussian_mixture(window_standardized_df, threshold= 1 - 10 ** (-16))
# Print the labeled DataFrame


In [ ]:
# Count True and False values in the "LEAK" column
leak_counts = labeled_df["LEAK"].value_counts()
# Print the counts
print(leak_counts)

In [ ]:
columns = [f'FLOW_{i}' for i in range(1, 4 + 1)]
X = window_standardized_df[columns]

X

### Testing

#### Calculate the mean flow for each hour, housing, usage and plot it

In [ ]:
from itertools import product
average_per_hour_usage = filtered_df.groupby(['HOUSING', 'USAGE','HOUR'])['FLOW'].mean().reset_index()

# Get unique housing and usage types
housing_types = average_per_hour_usage['HOUSING'].unique()
usage_types = average_per_hour_usage['USAGE'].unique()

# Create a generator for all combinations
all_combinations = product(housing_types, usage_types)

fig, axs = plt.subplots(len(housing_types), len(usage_types), figsize=(12, 20))  # Adjust size as needed

# Loop through all combinations
for i, (housing, usage) in enumerate(all_combinations):
    row, col = divmod(i, len(usage_types))  # Calculate row and column based on index
    data = average_per_hour_usage[
        (average_per_hour_usage['HOUSING'] == housing) & (average_per_hour_usage['USAGE'] == usage)
    ]
    ax = axs[row, col]
    ax.bar(data['HOUR'], data['FLOW'])
    ax.set_xlabel('Hour')
    ax.set_ylabel('Flow')
    ax.set_title(f'Flow for {housing} ({usage})')

plt.tight_layout()
plt.show()

In [ ]:
average_per_hour_usage = filtered_df.groupby(['USAGE', 'HOUR'])['FLOW'].mean().reset_index()
# Assuming you have a DataFrame 'average_per_hour_usage' with columns 'HOUR', 'USAGE', and 'FLOW'
usage_types = average_per_hour_usage['USAGE'].unique()

fig, axs = plt.subplots(2, 2, figsize=(10, 8))

for i, usage in enumerate(usage_types):
  data = average_per_hour_usage[average_per_hour_usage['USAGE'] == usage]
  ax = axs[i // 2, i % 2]  # Assign subplot to each usage type
  ax.bar(data['HOUR'], data['FLOW'])  # Use bar function for bar graph
  ax.set_xlabel('Hour')
  ax.set_ylabel('Flow')
  ax.set_title(f'Flow for {usage}')

plt.tight_layout()
plt.show()

# ----

#### Old code

In [ ]:
def setup_input(df, window_size):
    # Create the sliding window dataset
    result = []
    for policy in df['POLICY'].unique():
        policy_data = df[df['POLICY'] == policy]
        flows = policy_data['FLOW'].values
        technology = policy_data['TECHNOLOGY'].values
        usage = policy_data['USAGE'].values
        housing = policy_data['HOUSING'].values
        consumption = policy_data['CONSUMPTION'].values
        hour_date = policy_data['DATETIME'].values
        weekday = policy_data['WEEKDAY'].values
        hours = policy_data['HOUR'].values

        # Iterate over the range to capture each window of the specified size
        for i in range(len(flows) - window_size + 1):
            window_data = {'POLICY': policy, 'TECHNOLOGY': technology[i], 'USAGE': usage[i],
                           'HOUSING': housing[i],'CONSUMPTION': consumption[i], 'DATETIME': hour_date[i],
                           'WEEKDAY': weekday[i],'START_HOUR': hours[i]}  # Include starting hour
            for j in range(window_size):
                window_data[f'FLOW_{j+1}'] = flows[i + j]
            result.append(window_data)

    # Convert the result to a DataFrame
    return pd.DataFrame(result)

In [ ]:
def group_by_policy_and_sliding_window(df, window_size=10):
  """
  Groups data by policy and joins consecutive hours using a sliding window.

  Args:
    df: Pandas DataFrame with 'POLICY', 'WEEKDAY', 'HOUR', and 'CONSUMPTION' columns.
    window_size: Size of the sliding window (number of consecutive hours to join).

  Returns:
    A new DataFrame with grouped and joined data.
  """

  def join_consecutive_hours(group):
    """
    Joins consecutive hours within each group using a sliding window.
    """
    joined_data = []
    for i in range(len(group) - window_size + 1):
      window = group.iloc[i : i + window_size]
      joined_row = {
          'POLICY': window['POLICY'].iloc[0],
          'WEEKDAY': window['WEEKDAY'].iloc[0],
          'HOUR_START': window['HOUR'].iloc[0],
          'FLOW_1': window['FLOW'].iloc[0],
          'FLOW_2': window['FLOW'].iloc[1],
          'FLOW_3': window['FLOW'].iloc[2],
          'FLOW_4': window['FLOW'].iloc[3],
          'FLOW_5': window['FLOW'].iloc[4],
          'FLOW_6': window['FLOW'].iloc[5],
          'FLOW_7': window['FLOW'].iloc[6],
          'FLOW_8': window['FLOW'].iloc[7],
          'FLOW_9': window['FLOW'].iloc[8],
          'FLOW_10': window['FLOW'].iloc[9],
          # Add more CONSUMPTION_x columns as needed based on window_size
      }
      joined_data.append(joined_row)
    return pd.DataFrame(joined_data)

  # Group by 'POLICY'
  grouped = df.groupby('POLICY')

  # Apply the join_consecutive_hours function to each group
  joined_df = grouped.apply(join_consecutive_hours).reset_index(drop=True)

  return joined_df

# Assuming you have a DataFrame 'df' with the necessary columns

# Group and join data
# joined_df = group_by_policy_and_sliding_window(filtered_df, window_size=3)

# Print the joined DataFrame
# new_filtered_df.mean()

In [ ]:
def join_hourly_averages_in_triplets(average_df):
  """
  Joins hourly average consumption data in triplets of consecutive hours.

  Args:
    average_df: Pandas DataFrame with 'POLICY', 'USAGE', 'HOUSING', 
                'WEEKDAY', 'HOUR', and 'FLOW' columns, representing 
                hourly average consumption.

  Returns:
    A new DataFrame with joined triplets of hourly averages.
  """

  def join_consecutive_hours(group):
    """
    Joins consecutive hours within each group into triplets.
    """
    joined_data = []
    for i in range(len(group) - 3):  # Adjust for triplets
      window = group.iloc[i : i + 4]  # Window of 3 hours
      joined_row = {
          'POLICY' : window['POLICY'].iloc[0],
          'USAGE': window['USAGE'].iloc[0],
          'HOUSING': window['HOUSING'].iloc[0],
          'WEEKDAY': window['WEEKDAY'].iloc[0],
          'HOUR_START': window['HOUR'].iloc[0],  # Starting hour of the triplet
          'FLOW_1': window['FLOW'].iloc[0],
          'FLOW_2': window['FLOW'].iloc[1],
          'FLOW_3': window['FLOW'].iloc[2],
          'FLOW_4': window['FLOW'].iloc[3],
      }
      joined_data.append(joined_row)
    return pd.DataFrame(joined_data)

  # Group by 'POLICY', 'USAGE', 'HOUSING', and 'WEEKDAY'
  grouped = average_df.groupby(['USAGE', 'HOUSING', 'WEEKDAY'])

  # Apply the join_consecutive_hours function to each group
  joined_df = grouped.apply(join_consecutive_hours).reset_index(drop=True)

  return joined_df

In [ ]:
joined_averages_df = group_by_policy_and_sliding_window(filtered_df)
joined_averages_df

In [ ]:
average = filtered_df.groupby(['POLICY', 'USAGE', 'HOUSING', 'WEEKDAY', 'HOUR'])['FLOW'].mean().reset_index()
average_without_policy = filtered_df.groupby(['USAGE', 'HOUSING', 'WEEKDAY', 'HOUR'])['FLOW'].mean().reset_index()
joined_averages_df = join_hourly_averages_in_triplets(average_without_policy)
print(len(joined_averages_df['POLICY'].unique()))

In [ ]:
new_filtered_df = [] # Avoid warning

new_filtered_df['WEEKDAY'].unique()
print(len(new_filtered_df['POLICY'].unique()))

len(filtered_df["POLICY"])
len(joined_averages_df["POLICY"])

### Deprecated

Outlier filtering is not necessary anymore since we are using the housing aswell.

In [ ]:
ajuntament_flow_greater_than_4000 = filtered_df.loc[(filtered_df['USAGE'] == 'AJUNTAMENT') & (filtered_df['FLOW'] > 4000)]['POLICY'].values
new_filtered_df = filtered_df[~filtered_df['POLICY'].isin(ajuntament_flow_greater_than_4000)]

new_filtered_df

We dont need to set up the mean

In [ ]:
def setup_mean(mean, window_size):
    # Create the sliding window dataset
    result = []
    
    # Get unique combinations of 'USAGE', 'HOUSING', 'WEEKDAY'
    unique_combinations = mean[['USAGE', 'HOUSING', 'WEEKDAY']].drop_duplicates()
    for _, row in unique_combinations.iterrows():
        combo = mean[(mean['USAGE'] == row['USAGE']) & (mean['HOUSING'] == row['HOUSING']) & (mean['WEEKDAY'] == row['WEEKDAY'])]
        flows = combo['FLOW'].reset_index(drop=True)
        hours = combo['HOUR'].reset_index(drop=True)
        
        # Iterate over the range to capture each window of the specified size
        for i in range(len(flows) - window_size + 1):
            window_data = {
                'USAGE': row['USAGE'],
                    'HOUSING': row['HOUSING'],
                'WEEKDAY': row['WEEKDAY'],
                'START_HOUR': hours.iloc[i]  # Access by position
            }
            for j in range(window_size):
                window_data[f'FLOW_{j+1}'] = flows.iloc[i + j]  # Access by position
            result.append(window_data)
    
    # Convert the result to a DataFrame
    return pd.DataFrame(result)
